# KNN Titanic

Clasificacion KNN para supervivencia en Titanic.

Conversion conceptual 1:1 desde el ejemplo R homologo.


In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "R").exists():
            return candidate
    raise FileNotFoundError("No se encontro la carpeta R del repositorio")

REPO_ROOT = find_repo_root(Path.cwd())
print("Repo root:", REPO_ROOT)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

data_path = REPO_ROOT / "R" / "nuevos" / "5_aprendizaje_supervisado" / "data" / "titanic.csv"
df = pd.read_csv(data_path)
y = df["Survived"]
X = df[["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]

num_cols = ["Age", "Fare", "SibSp", "Parch", "Pclass"]
cat_cols = ["Sex", "Embarked"]
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model = Pipeline([("pre", pre), ("clf", KNeighborsClassifier(n_neighbors=7))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))
